<a href="https://colab.research.google.com/github/JorgeMiceli1967/IA-SCRAPPING/blob/main/RELEVAMIENTO_DUCK_G.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# RELEVAMIENTO WEB TEMÁTICO PARA ESTADO DEL ARTE
# VERSIÓN COLAB - DDGS (DuckDuckGo)
# BÚSQUEDA LIBRE + BÚSQUEDA FORZADA EN REPOSITORIOS
# TODO RESUELTO EN UN SOLO SCRIPT
# ============================================================

import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")

import sys
import subprocess
import re
import time
from urllib.parse import urlparse, urlunparse

# ---------- instalar paquetes ----------
def ensure_package(package_name: str):
    try:
        __import__(package_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", package_name])

ensure_package("ddgs")
ensure_package("openpyxl")
ensure_package("pandas")

import pandas as pd
from ddgs import DDGS

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

RESULTS_PER_QUERY = 10
SLEEP_BETWEEN_SEARCHES = 1.0
MAX_BASE_QUERIES = None

OUTPUT_CSV = "relevamiento_estado_del_arte.csv"
OUTPUT_XLSX = "relevamiento_estado_del_arte.xlsx"

# ------------------------------------------------------------
# DESDUPLICACIÓN DE URLS
# ------------------------------------------------------------
DEDUPLICATE_URLS = True
URL_DEDUP_STRATEGY = "canonical"   # "exact" o "canonical"

# ------------------------------------------------------------
# MODO DE COMBINACIÓN DE KEYWORDS
# ------------------------------------------------------------
# True  -> producto cartesiano absoluto entre keywords_juridicas y keywords_informaticas
# False -> usa el PAIR_MAP
USE_ABSOLUTE_CARTESIAN_PRODUCT = True

# Cada buscador lógico puede:
# - hacer búsqueda libre
# - hacer búsqueda forzada en repositorios concretos con site:
SEARCH_ENGINES = {
    "BIBLIO_ACADEMICA": {
        "campo": "Bibliografía",
        "required_sites": [
            "dialnet.unirioja.es",
            "scholar.google.com",
            "editorial.tirant.com/es",
            "tienda.thomsonreuters.com.ar"
        ],
        "include_free_search": True,
    },
}

# ============================================================
# PARES CONTROLADOS Y LISTAS DE KEYWORDS
# ============================================================

PAIR_MAP = {
    "inteligencia artificial": [
        "derecho penal",
        "proceso penal",
        "investigación penal",
        "prueba penal",
        "prueba pericial",
        "inferencia probatoria",
    ],
    "machine learning": [
        "investigación penal",
        "prueba pericial",
        "criminalistica",
        "inferencia probatoria",
    ],
    "datos": [
        "derecho penal",
        "investigación penal",
        "prueba penal",
        "criminalistica",
    ],
    "digital": [
        "proceso penal",
        "procedimiento penal",
        "investigación penal",
    ],
    "big data": [
        "investigación penal",
        "criminalistica",
        "inferencia probatoria",
    ],
    "agente inteligente": [
        "proceso penal",
        "prueba penal",
    ],
}

keywords_juridicas = [
    "derecho penal", "proceso penal", "procedimiento penal", "investigación penal",
    "prueba penal", "inferencia penal", "hipotesis penal", "teoría caso",
    "prueba pericial", "prueba caso", "peritaje", "forense",
    "criminalistica", "inferencia probatoria", "acusacion", "defensa"
]

keywords_informaticas = [
    "IA", "AI", "inteligencia artificial", "datos", "digital",
    "machine learning", "deep learning", "big data", "datamining",
    "agentic workflow", "agente inteligente", "fichero red",
    "archivo computarizado", "registro informatico",
    "informacion electronica", "digitalizado", "teconologia", "tecnologico"
]

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def normalize_space(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def extract_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""

def canonicalize_url(url: str) -> str:
    try:
        parsed = urlparse(normalize_space(url))
        scheme = (parsed.scheme or "https").lower()
        netloc = parsed.netloc.lower().replace("www.", "")
        path = parsed.path or "/"

        if path != "/":
            path = path.rstrip("/")
            if not path.startswith("/"):
                path = "/" + path

        raw_query = parsed.query.strip()
        query_pairs = []
        if raw_query:
            for part in raw_query.split("&"):
                if "=" in part:
                    k, v = part.split("=", 1)
                else:
                    k, v = part, ""
                k = k.strip()
                v = v.strip()

                if k.lower().startswith("utm_"):
                    continue
                if k.lower() in {"fbclid", "gclid", "igshid", "mc_cid", "mc_eid"}:
                    continue

                query_pairs.append((k, v))

        query_pairs = sorted(query_pairs, key=lambda x: (x[0], x[1]))
        normalized_query = "&".join(
            [f"{k}={v}" if v != "" else k for k, v in query_pairs]
        )

        return urlunparse((scheme, netloc, path, "", normalized_query, ""))
    except Exception:
        return normalize_space(url)

def get_dedup_key(url: str) -> str:
    if URL_DEDUP_STRATEGY == "canonical":
        return canonicalize_url(url)
    return normalize_space(url)

def classify_actor_type(domain: str, text: str) -> str:
    d = (domain or "").lower()
    t = (text or "").lower()

    if any(x in d for x in [".gov", ".gob", "fiscal", "judiciary", "justice", "poderjudicial", "boe.es"]):
        return "gobierno/justicia"

    if any(x in d for x in [".edu", ".ac.", "universidad", "university"]) or \
       any(x in t for x in ["universidad", "university", "facultad", "faculty", "research group", "grupo de investigación"]):
        return "academia"

    if any(x in d for x in [".org", ".int"]) or \
       any(x in t for x in ["ngo", "ong", "nonprofit", "asociación", "fundación", "foundation"]):
        return "ONG"

    if any(x in t for x in ["software", "platform", "plataforma", "startup", "company", "empresa", "provider"]) or \
       any(x in d for x in [".com", ".io", ".ai"]):
        return "empresa"

    return "otro"

def build_base_queries_from_pair_map(pair_map: dict, max_queries: int | None = None) -> list[str]:
    queries = []
    for kw_info, juridicas in pair_map.items():
        for kw_jur in juridicas:
            queries.append(normalize_space(f"{kw_info} {kw_jur}"))
    queries = sorted(set(queries))
    if max_queries is not None:
        queries = queries[:max_queries]
    return queries

def build_base_queries_cartesian(
    keywords_juridicas: list[str],
    keywords_informaticas: list[str],
    max_queries: int | None = None
) -> list[str]:
    queries = []
    for kw_jur in keywords_juridicas:
        for kw_info in keywords_informaticas:
            queries.append(normalize_space(f"{kw_info} {kw_jur}"))
    queries = sorted(set(queries))
    if max_queries is not None:
        queries = queries[:max_queries]
    return queries

def build_base_queries(
    use_absolute_cartesian_product: bool,
    pair_map: dict,
    keywords_juridicas: list[str],
    keywords_informaticas: list[str],
    max_queries: int | None = None
) -> list[str]:
    if use_absolute_cartesian_product:
        return build_base_queries_cartesian(
            keywords_juridicas=keywords_juridicas,
            keywords_informaticas=keywords_informaticas,
            max_queries=max_queries
        )

    return build_base_queries_from_pair_map(
        pair_map=pair_map,
        max_queries=max_queries
    )

def compose_engine_queries(
    required_sites: list[str],
    base_query: str,
    include_free_search: bool = False
) -> list[str]:
    queries = []

    if include_free_search:
        queries.append(base_query)

    for site in required_sites:
        queries.append(f"{base_query} site:{site}")

    return sorted(set(queries))

def format_progress(current: int, total: int) -> str:
    if total <= 0:
        return "0.00%"
    return f"{(current / total) * 100:.2f}%"

# ============================================================
# BÚSQUEDA DDGS
# ============================================================

def search_ddgs(query: str, max_results: int = 10) -> list[dict]:
    results = []
    with DDGS() as ddgs:
        raw = ddgs.text(query, max_results=max_results)
        for item in raw:
            title = normalize_space(item.get("title", ""))
            link = normalize_space(item.get("href", ""))
            snippet = normalize_space(item.get("body", ""))
            if title and link:
                results.append({
                    "title": title,
                    "link": link,
                    "snippet": snippet,
                })
    return results

def normalize_search_result(field_name: str, query: str, item: dict) -> dict:
    url = normalize_space(item.get("link", ""))
    title = normalize_space(item.get("title", ""))
    snippet = normalize_space(item.get("snippet", ""))
    domain = extract_domain(url)

    page_title = title
    combined_text = " ".join([title, snippet, page_title])
    actor_type = classify_actor_type(domain, combined_text)

    return {
        "CAMPO": field_name,
        "ACTOR": actor_type,
        "TÍTULO": title,
        "RESUMEN BÚSQUEDA": snippet,
        "TÍTULO PÁGINA": page_title,
        "DOMINIO": domain,
        "URL": url,
        "URL_NORMALIZADA": get_dedup_key(url),
        "CONSULTA": query,
    }

# ============================================================
# TEST INICIAL
# ============================================================

def test_search():
    test_query = "derecho penal inteligencia artificial site:dialnet.unirioja.es"
    print("=== TEST INICIAL DDGS ===")
    print("Query:", test_query)
    results = search_ddgs(test_query, max_results=3)
    print("Resultados de prueba:", len(results))
    if results:
        print("Primer título:", results[0]["title"])
        print("Primera URL:", results[0]["link"])
    print()

# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    test_search()

    base_queries = build_base_queries(
        use_absolute_cartesian_product=USE_ABSOLUTE_CARTESIAN_PRODUCT,
        pair_map=PAIR_MAP,
        keywords_juridicas=keywords_juridicas,
        keywords_informaticas=keywords_informaticas,
        max_queries=MAX_BASE_QUERIES
    )

    print("=== RESUMEN ===")
    print("Modo de combinación:",
          "producto cartesiano absoluto" if USE_ABSOLUTE_CARTESIAN_PRODUCT else "PAIR_MAP")
    print("Base queries generadas:", len(base_queries))
    print(f"Desduplicación de URLs: {'sí' if DEDUPLICATE_URLS else 'no'}")
    print(f"Estrategia de desduplicación: {URL_DEDUP_STRATEGY}")
    print()

    rows = []

    for engine_name, engine_conf in SEARCH_ENGINES.items():
        campo = engine_conf["campo"]
        required_sites = engine_conf.get("required_sites", [])
        include_free_search = engine_conf.get("include_free_search", False)

        print(f"=== Buscador lógico: {engine_name} ===")
        print(f"Campo: {campo}")
        print(f"Búsqueda libre: {'sí' if include_free_search else 'no'}")
        print(f"Repositorios forzados: {', '.join(required_sites) if required_sites else '[ninguno]'}")
        print()

        engine_queries = []
        for base_query in base_queries:
            engine_queries.extend(
                compose_engine_queries(
                    required_sites=required_sites,
                    base_query=base_query,
                    include_free_search=include_free_search
                )
            )

        engine_queries = sorted(set(engine_queries))
        total_queries = len(engine_queries)

        print(f"Queries efectivas en este buscador: {total_queries}")
        print()

        for idx, final_query in enumerate(engine_queries, start=1):
            progress_pct = format_progress(idx, total_queries)
            print(f"[{idx}/{total_queries}] ({progress_pct}) {final_query}")

            try:
                results = search_ddgs(final_query, max_results=RESULTS_PER_QUERY)

                for item in results:
                    row = normalize_search_result(campo, final_query, item)
                    if row["URL"]:
                        rows.append(row)

            except Exception as e:
                print(f"Error en '{final_query}' ({engine_name}): {e}")

            time.sleep(SLEEP_BETWEEN_SEARCHES)

        print()

    df = pd.DataFrame(rows)

    if df.empty:
        print("No se recuperaron resultados.")
        return

    total_before_dedup = len(df)

    if DEDUPLICATE_URLS:
        df = df.drop_duplicates(subset=["URL_NORMALIZADA"], keep="first").reset_index(drop=True)

    total_after_dedup = len(df)

    ordered_cols = [
        "CAMPO",
        "ACTOR",
        "TÍTULO",
        "RESUMEN BÚSQUEDA",
        "TÍTULO PÁGINA",
        "DOMINIO",
        "URL",
        "CONSULTA",
    ]

    df = df.sort_values(
        by=["CAMPO", "CONSULTA", "ACTOR", "DOMINIO", "TÍTULO"],
        ascending=[True, True, True, True, True],
    ).reset_index(drop=True)

    print("=== RESUMEN DE DUPLICADOS ===")
    print("Filas antes de desduplicar:", total_before_dedup)
    print("Filas después de desduplicar:", total_after_dedup)
    print("Duplicados eliminados:", total_before_dedup - total_after_dedup)
    print()

    print("=== VISTA PREVIA ===")
    print(df[ordered_cols].head(15).to_string(index=False))

    df[ordered_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df[ordered_cols].to_excel(writer, sheet_name="resultados", index=False)

    print()
    print("Archivos guardados:")
    print("-", OUTPUT_CSV)
    print("-", OUTPUT_XLSX)

    try:
        from google.colab import files
        files.download(OUTPUT_CSV)
        files.download(OUTPUT_XLSX)
    except Exception:
        pass

main()

=== TEST INICIAL DDGS ===
Query: derecho penal inteligencia artificial site:dialnet.unirioja.es
Resultados de prueba: 3
Primer título: DerechoPenal,InteligenciaArtificialy Neurociencias - Dialnet
Primera URL: https://dialnet.unirioja.es/servlet/libro?codigo=912449

=== RESUMEN ===
Modo de combinación: producto cartesiano absoluto
Base queries generadas: 288
Desduplicación de URLs: sí
Estrategia de desduplicación: canonical

=== Buscador lógico: BIBLIO_ACADEMICA ===
Campo: Bibliografía
Búsqueda libre: sí
Repositorios forzados: dialnet.unirioja.es, scholar.google.com, editorial.tirant.com/es, tienda.thomsonreuters.com.ar

Queries efectivas en este buscador: 1440

[1/1440] (0.07%) AI acusacion
[2/1440] (0.14%) AI acusacion site:dialnet.unirioja.es
[3/1440] (0.21%) AI acusacion site:editorial.tirant.com/es
[4/1440] (0.28%) AI acusacion site:scholar.google.com
[5/1440] (0.35%) AI acusacion site:tienda.thomsonreuters.com.ar
Error en 'AI acusacion site:tienda.thomsonreuters.com.ar' (BIBLIO_AC

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# SISTEMA DE RELEVAMIENTO WEB TEMÁTICO PARA ESTADO DEL ARTE
# ------------------------------------------------------------
# Versión: Colab / Python standalone
# Motor de búsqueda: DDGS (DuckDuckGo Search API no oficial)
#
# DESCRIPCIÓN GENERAL
# ------------------------------------------------------------
# Este script implementa un pipeline automatizado para la
# construcción de corpus documental orientado a revisiones
# sistemáticas y estados del arte en dominios interdisciplinarios.
#
# Permite generar consultas de búsqueda a partir de:
#   (a) un mapeo dirigido entre categorías conceptuales (PAIR_MAP)
#   (b) un producto cartesiano completo entre listas de keywords
#       (por ejemplo, dominio jurídico × dominio informático)
#
# Las consultas se ejecutan de manera:
#   - libre (búsqueda abierta)
#   - restringida a repositorios específicos mediante operadores site:
#
# FUNCIONALIDADES PRINCIPALES
# ------------------------------------------------------------
# - Generación sistemática y configurable de queries
# - Integración de búsqueda libre y búsqueda en fuentes priorizadas
# - Extracción de resultados (título, snippet, URL)
# - Normalización y estructuración de datos
# - Clasificación heurística del tipo de actor (academia, Estado, empresa, etc.)
# - Desduplicación de URLs (estrategia exacta o canónica)
# - Monitoreo de progreso en tiempo real (% de avance)
# - Exportación a CSV y Excel
#
# VARIABLES CONFIGURABLES CLAVE
# ------------------------------------------------------------
# - Modo de combinación de keywords (cartesiano vs dirigido)
# - Cantidad de resultados por consulta
# - Límite de queries generadas
# - Repositorios forzados (site:)
# - Inclusión o no de búsqueda libre
# - Activación y estrategia de desduplicación
# - Frecuencia de consultas (delay entre requests)
#
# SALIDA
# ------------------------------------------------------------
# Dataset estructurado con:
#   CAMPO, ACTOR, TÍTULO, RESUMEN, DOMINIO, URL, CONSULTA
#
# Uso típico:
#   - construcción de estados del arte
#   - relevamientos exploratorios
#   - identificación de actores institucionales
#   - análisis preliminar de literatura y fuentes
#
# NOTA
# ------------------------------------------------------------
# El script está diseñado para ser auto-contenido, reproducible
# y fácilmente extensible (nuevos dominios, nuevos repositorios,
# nuevas heurísticas de clasificación o filtrado).
# ============================================================

import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")

import sys
import subprocess
import re
import time
from urllib.parse import urlparse, urlunparse

# ---------- instalar paquetes ----------
def ensure_package(package_name: str):
    try:
        __import__(package_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", package_name])

ensure_package("ddgs")
ensure_package("openpyxl")
ensure_package("pandas")

import pandas as pd
from ddgs import DDGS

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

RESULTS_PER_QUERY = 10
SLEEP_BETWEEN_SEARCHES = 1.0
MAX_BASE_QUERIES = 300

OUTPUT_CSV = "relevamiento_estado_del_arte.csv"
OUTPUT_XLSX = "relevamiento_estado_del_arte.xlsx"

# ------------------------------------------------------------
# DESDUPLICACIÓN DE URLS
# ------------------------------------------------------------
DEDUPLICATE_URLS = True
URL_DEDUP_STRATEGY = "canonical"   # "exact" o "canonical"

# ------------------------------------------------------------
# MODO DE COMBINACIÓN DE KEYWORDS
# ------------------------------------------------------------
# True  -> producto cartesiano absoluto entre keywords_juridicas y keywords_informaticas
# False -> usa el PAIR_MAP
USE_ABSOLUTE_CARTESIAN_PRODUCT = True

# Cada buscador lógico puede:
# - hacer búsqueda libre
# - hacer búsqueda forzada en repositorios concretos con site:
SEARCH_ENGINES = {
    "BIBLIO_ACADEMICA": {
        "campo": "Bibliografía",
        "required_sites": [
            "dialnet.unirioja.es",
            "scholar.google.com",
            "editorial.tirant.com/es",
            "tienda.thomsonreuters.com.ar"
        ],
        "include_free_search": True,
    },
}

# ============================================================
# PARES CONTROLADOS Y LISTAS DE KEYWORDS
# ============================================================

PAIR_MAP = {
    "inteligencia artificial": [
        "derecho penal",
        "proceso penal",
        "investigación penal",
        "prueba penal",
        "prueba pericial",
        "inferencia probatoria",
    ],
    "machine learning": [
        "investigación penal",
        "prueba pericial",
        "criminalistica",
        "inferencia probatoria",
    ],
    "datos": [
        "derecho penal",
        "investigación penal",
        "prueba penal",
        "criminalistica",
    ],
    "digital": [
        "proceso penal",
        "procedimiento penal",
        "investigación penal",
    ],
    "big data": [
        "investigación penal",
        "criminalistica",
        "inferencia probatoria",
    ],
    "agente inteligente": [
        "proceso penal",
        "prueba penal",
    ],
}

keywords_juridicas = [
    "derecho penal", "proceso penal", "procedimiento penal", "investigación penal",
    "prueba penal", "inferencia penal", "hipotesis penal", "teoría caso",
    "prueba pericial", "prueba caso", "peritaje", "forense",
    "criminalistica", "inferencia probatoria", "acusacion", "defensa"
]

keywords_informaticas = [
    "IA", "AI", "inteligencia artificial", "datos", "digital",
    "machine learning", "deep learning", "big data", "datamining",
    "agentic workflow", "agente inteligente", "fichero red",
    "archivo computarizado", "registro informatico",
    "informacion electronica", "digitalizado", "teconologia", "tecnologico"
]

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def normalize_space(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def extract_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""

def canonicalize_url(url: str) -> str:
    try:
        parsed = urlparse(normalize_space(url))
        scheme = (parsed.scheme or "https").lower()
        netloc = parsed.netloc.lower().replace("www.", "")
        path = parsed.path or "/"

        if path != "/":
            path = path.rstrip("/")
            if not path.startswith("/"):
                path = "/" + path

        raw_query = parsed.query.strip()
        query_pairs = []
        if raw_query:
            for part in raw_query.split("&"):
                if "=" in part:
                    k, v = part.split("=", 1)
                else:
                    k, v = part, ""
                k = k.strip()
                v = v.strip()

                if k.lower().startswith("utm_"):
                    continue
                if k.lower() in {"fbclid", "gclid", "igshid", "mc_cid", "mc_eid"}:
                    continue

                query_pairs.append((k, v))

        query_pairs = sorted(query_pairs, key=lambda x: (x[0], x[1]))
        normalized_query = "&".join(
            [f"{k}={v}" if v != "" else k for k, v in query_pairs]
        )

        return urlunparse((scheme, netloc, path, "", normalized_query, ""))
    except Exception:
        return normalize_space(url)

def get_dedup_key(url: str) -> str:
    if URL_DEDUP_STRATEGY == "canonical":
        return canonicalize_url(url)
    return normalize_space(url)

def classify_actor_type(domain: str, text: str) -> str:
    d = (domain or "").lower()
    t = (text or "").lower()

    if any(x in d for x in [".gov", ".gob", "fiscal", "judiciary", "justice", "poderjudicial", "boe.es"]):
        return "gobierno/justicia"

    if any(x in d for x in [".edu", ".ac.", "universidad", "university"]) or \
       any(x in t for x in ["universidad", "university", "facultad", "faculty", "research group", "grupo de investigación"]):
        return "academia"

    if any(x in d for x in [".org", ".int"]) or \
       any(x in t for x in ["ngo", "ong", "nonprofit", "asociación", "fundación", "foundation"]):
        return "ONG"

    if any(x in t for x in ["software", "platform", "plataforma", "startup", "company", "empresa", "provider"]) or \
       any(x in d for x in [".com", ".io", ".ai"]):
        return "empresa"

    return "otro"

def infer_author_from_snippet(snippet: str) -> str:
    if not snippet:
        return ""

    patterns = [
        r"(?:por|autor|autora)\s*:\s*([A-ZÁÉÍÓÚÑ][^.,;:\-\|]{3,80})",
        r"(?:por)\s+([A-ZÁÉÍÓÚÑ][^.,;:\-\|]{3,80})",
        r"(?:By)\s+([A-Z][^.,;:\-\|]{3,80})",
    ]

    for pat in patterns:
        m = re.search(pat, snippet, flags=re.IGNORECASE)
        if m:
            return normalize_space(m.group(1))

    return ""

def build_base_queries_from_pair_map(pair_map: dict, max_queries: int | None = None) -> list[str]:
    queries = []
    for kw_info, juridicas in pair_map.items():
        for kw_jur in juridicas:
            queries.append(normalize_space(f"{kw_info} {kw_jur}"))
    queries = sorted(set(queries))
    if max_queries is not None:
        queries = queries[:max_queries]
    return queries

def build_base_queries_cartesian(
    keywords_juridicas: list[str],
    keywords_informaticas: list[str],
    max_queries: int | None = None
) -> list[str]:
    queries = []
    for kw_jur in keywords_juridicas:
        for kw_info in keywords_informaticas:
            queries.append(normalize_space(f"{kw_info} {kw_jur}"))
    queries = sorted(set(queries))
    if max_queries is not None:
        queries = queries[:max_queries]
    return queries

def build_base_queries(
    use_absolute_cartesian_product: bool,
    pair_map: dict,
    keywords_juridicas: list[str],
    keywords_informaticas: list[str],
    max_queries: int | None = None
) -> list[str]:
    if use_absolute_cartesian_product:
        return build_base_queries_cartesian(
            keywords_juridicas=keywords_juridicas,
            keywords_informaticas=keywords_informaticas,
            max_queries=max_queries
        )

    return build_base_queries_from_pair_map(
        pair_map=pair_map,
        max_queries=max_queries
    )

def compose_engine_queries(
    required_sites: list[str],
    base_query: str,
    include_free_search: bool = False
) -> list[str]:
    queries = []

    if include_free_search:
        queries.append(base_query)

    for site in required_sites:
        queries.append(f"{base_query} site:{site}")

    return sorted(set(queries))

# ============================================================
# BÚSQUEDA DDGS
# ============================================================

def search_ddgs(query: str, max_results: int = 10) -> list[dict]:
    results = []
    with DDGS() as ddgs:
        raw = ddgs.text(query, max_results=max_results)
        for item in raw:
            title = normalize_space(item.get("title", ""))
            link = normalize_space(item.get("href", ""))
            snippet = normalize_space(item.get("body", ""))
            if title and link:
                results.append({
                    "title": title,
                    "link": link,
                    "snippet": snippet,
                })
    return results

def normalize_search_result(field_name: str, query: str, item: dict) -> dict:
    url = normalize_space(item.get("link", ""))
    title = normalize_space(item.get("title", ""))
    snippet = normalize_space(item.get("snippet", ""))
    domain = extract_domain(url)
    author = infer_author_from_snippet(snippet)

    page_title = title
    combined_text = " ".join([title, snippet, page_title, author])
    actor_type = classify_actor_type(domain, combined_text)

    return {
        "CAMPO": field_name,
        "ACTOR": actor_type,
        "AUTOR": author,
        "TÍTULO": title,
        "RESUMEN BÚSQUEDA": snippet,
        "TÍTULO PÁGINA": page_title,
        "DOMINIO": domain,
        "URL": url,
        "URL_NORMALIZADA": get_dedup_key(url),
        "CONSULTA": query,
    }

# ============================================================
# TEST INICIAL
# ============================================================

def test_search():
    test_query = "derecho penal inteligencia artificial site:dialnet.unirioja.es"
    print("=== TEST INICIAL DDGS ===")
    print("Query:", test_query)
    results = search_ddgs(test_query, max_results=3)
    print("Resultados de prueba:", len(results))
    if results:
        print("Primer título:", results[0]["title"])
        print("Primera URL:", results[0]["link"])
    print()

# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    test_search()

    base_queries = build_base_queries(
        use_absolute_cartesian_product=USE_ABSOLUTE_CARTESIAN_PRODUCT,
        pair_map=PAIR_MAP,
        keywords_juridicas=keywords_juridicas,
        keywords_informaticas=keywords_informaticas,
        max_queries=MAX_BASE_QUERIES
    )

    print("=== RESUMEN ===")
    print("Modo de combinación:",
          "producto cartesiano absoluto" if USE_ABSOLUTE_CARTESIAN_PRODUCT else "PAIR_MAP")
    print("Base queries generadas:", len(base_queries))
    print(f"Desduplicación de URLs: {'sí' if DEDUPLICATE_URLS else 'no'}")
    print(f"Estrategia de desduplicación: {URL_DEDUP_STRATEGY}")
    print()

    rows = []

    for engine_name, engine_conf in SEARCH_ENGINES.items():
        campo = engine_conf["campo"]
        required_sites = engine_conf.get("required_sites", [])
        include_free_search = engine_conf.get("include_free_search", False)

        print(f"=== Buscador lógico: {engine_name} ===")
        print(f"Campo: {campo}")
        print(f"Búsqueda libre: {'sí' if include_free_search else 'no'}")
        print(f"Repositorios forzados: {', '.join(required_sites) if required_sites else '[ninguno]'}")
        print()

        engine_queries = []
        for base_query in base_queries:
            engine_queries.extend(
                compose_engine_queries(
                    required_sites=required_sites,
                    base_query=base_query,
                    include_free_search=include_free_search
                )
            )

        engine_queries = sorted(set(engine_queries))

        print(f"Queries efectivas en este buscador: {len(engine_queries)}")
        print()

        for idx, final_query in enumerate(engine_queries, start=1):
            print(f"[{idx}/{len(engine_queries)}] {final_query}")

            try:
                results = search_ddgs(final_query, max_results=RESULTS_PER_QUERY)

                for item in results:
                    row = normalize_search_result(campo, final_query, item)
                    if row["URL"]:
                        rows.append(row)

            except Exception as e:
                print(f"Error en '{final_query}' ({engine_name}): {e}")

            time.sleep(SLEEP_BETWEEN_SEARCHES)

        print()

    df = pd.DataFrame(rows)

    if df.empty:
        print("No se recuperaron resultados.")
        return

    total_before_dedup = len(df)

    if DEDUPLICATE_URLS:
        df = df.drop_duplicates(subset=["URL_NORMALIZADA"], keep="first").reset_index(drop=True)

    total_after_dedup = len(df)

    ordered_cols = [
        "CAMPO",
        "ACTOR",
        "AUTOR",
        "TÍTULO",
        "RESUMEN BÚSQUEDA",
        "TÍTULO PÁGINA",
        "DOMINIO",
        "URL",
        "CONSULTA",
    ]

    df = df.sort_values(
        by=["CAMPO", "CONSULTA", "ACTOR", "DOMINIO", "TÍTULO"],
        ascending=[True, True, True, True, True],
    ).reset_index(drop=True)

    print("=== RESUMEN DE DUPLICADOS ===")
    print("Filas antes de desduplicar:", total_before_dedup)
    print("Filas después de desduplicar:", total_after_dedup)
    print("Duplicados eliminados:", total_before_dedup - total_after_dedup)
    print()

    print("=== VISTA PREVIA ===")
    print(df[ordered_cols].head(15).to_string(index=False))

    df[ordered_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df[ordered_cols].to_excel(writer, sheet_name="resultados", index=False)

    print()
    print("Archivos guardados:")
    print("-", OUTPUT_CSV)
    print("-", OUTPUT_XLSX)

    try:
        from google.colab import files
        files.download(OUTPUT_CSV)
        files.download(OUTPUT_XLSX)
    except Exception:
        pass

main()

=== TEST INICIAL DDGS ===
Query: derecho penal inteligencia artificial site:dialnet.unirioja.es
Resultados de prueba: 3
Primer título: DerechoPenal,InteligenciaArtificialy Neurociencias - Dialnet
Primera URL: https://dialnet.unirioja.es/servlet/libro?codigo=912449

=== RESUMEN ===
Modo de combinación: producto cartesiano absoluto
Base queries generadas: 288
Desduplicación de URLs: sí
Estrategia de desduplicación: canonical

=== Buscador lógico: BIBLIO_ACADEMICA ===
Campo: Bibliografía
Búsqueda libre: sí
Repositorios forzados: dialnet.unirioja.es, scholar.google.com, editorial.tirant.com/es, tienda.thomsonreuters.com.ar

Queries efectivas en este buscador: 1440

[1/1440] AI acusacion
[2/1440] AI acusacion site:dialnet.unirioja.es
[3/1440] AI acusacion site:editorial.tirant.com/es
[4/1440] AI acusacion site:scholar.google.com
[5/1440] AI acusacion site:tienda.thomsonreuters.com.ar
[6/1440] AI criminalistica
[7/1440] AI criminalistica site:dialnet.unirioja.es
[8/1440] AI criminalistica si

KeyboardInterrupt: 